# Localization Evaluation (Optional)

## SymAD-ECNN Dissertation Evaluation

**Purpose**: Evaluate pixel-level anomaly localization accuracy using BraTS tumor masks as ground truth.

**Dissertation Chapter**: 8.5 (Further Evaluations - Localization Analysis)

This notebook provides:
1. Extraction of BraTS tumor masks as ground truth
2. Generation of reconstruction error maps from the ECNN model
3. Computation of pixel-level localization metrics (Dice, IoU, AUROC, etc.)
4. Visualization of localization quality

**Note**: This evaluation is optional and requires BraTS raw data with segmentation masks.

---

## 1. Environment Setup

In [ ]:
# Environment setup: mount Google Drive if running in Colab
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Running in Google Colab; Drive mounted at /content/drive")
except ImportError:
    IN_COLAB = False
    print("Running outside Colab; skipping Drive mount")

In [ ]:
# Add evals directory to path
import sys
from pathlib import Path

if IN_COLAB:
    EVALS_DIR = Path('/content/drive/MyDrive/symAD-ECNN/notebooks/evals')
else:
    EVALS_DIR = Path.cwd().parent if Path.cwd().name == 'localization' else Path.cwd()

sys.path.insert(0, str(EVALS_DIR))
sys.path.insert(0, str(EVALS_DIR / 'localization'))
sys.path.insert(0, str(EVALS_DIR / 'ecnn_thresholding'))

print(f"Evals directory: {EVALS_DIR}")

In [ ]:
# Import evaluation modules
from config import (
    EVALUATIONS_ROOT, ensure_directories_exist,
    DRIVE_PROJECT_ROOT
)
from path_utils import find_data_paths, get_drive_project_root
from io_utils import save_json, save_csv, log_message
from plotting_utils import save_figure

# Import localization modules
from extract_brats_mask_pairs import (
    extract_brats_mask_pairs,
    find_brats_volumes,
    load_mask_pair,
    get_all_mask_pairs,
    BRATS_REGIONS
)
from compute_pixel_metrics import (
    compute_all_pixel_metrics,
    compute_dice_coefficient,
    compute_iou,
    compute_reconstruction_error,
    evaluate_localization_batch,
    run_localization_evaluation,
    visualize_localization_result
)

print("Modules imported successfully.")

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime
import json

# Try to import nibabel
try:
    import nibabel as nib
    NIBABEL_AVAILABLE = True
except ImportError:
    NIBABEL_AVAILABLE = False
    print("Warning: nibabel not available")

# Configure matplotlib
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11

# Ensure directories
ensure_directories_exist()
print(f"Evaluations will be saved to: {EVALUATIONS_ROOT}")

## 2. Check Data Availability

In [ ]:
# Find BraTS raw data
data_paths = find_data_paths()

BRATS_DIR = None

# Check for brats2021 with segmentation masks
candidate_paths = [
    Path('/content/drive/MyDrive/symAD-ECNN/data/brats2021'),
    Path('/content/drive/MyDrive/data/brats2021'),
    get_drive_project_root() / 'data' / 'brats2021'
]

if data_paths.get('brats_raw'):
    candidate_paths.insert(0, Path(data_paths['brats_raw']))

for path in candidate_paths:
    if path.exists():
        # Check if it has segmentation files
        seg_files = list(path.glob('**/BraTS*_seg.nii.gz'))
        if seg_files:
            BRATS_DIR = path
            break

if BRATS_DIR:
    print(f"BraTS data found at: {BRATS_DIR}")
    volumes = find_brats_volumes(BRATS_DIR)
    print(f"Found {len(volumes)} patient volumes with segmentation masks.")
    BRATS_AVAILABLE = len(volumes) > 0
else:
    print("WARNING: BraTS raw data with segmentation masks not found.")
    print("This notebook requires BraTS 2021 data with tumor segmentation masks.")
    BRATS_AVAILABLE = False

In [ ]:
# Define output directory
MASK_PAIRS_DIR = EVALUATIONS_ROOT / 'localization' / 'mask_pairs'
MASK_PAIRS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Mask pairs will be saved to: {MASK_PAIRS_DIR}")

# Check if pairs already exist
existing_pairs = list(MASK_PAIRS_DIR.glob('*_t1.png'))
print(f"Existing pairs: {len(existing_pairs)}")

## 3. Extract Tumor Mask Pairs

Extract paired T1 images and tumor masks from BraTS data.

In [ ]:
# Configuration
EXTRACTION_CONFIG = {
    'max_patients': 10,           # Number of patients to process
    'slices_per_patient': 5,      # Slices per patient
    'region': 'whole_tumor',      # Tumor region to extract
}

print("Extraction Configuration:")
for key, value in EXTRACTION_CONFIG.items():
    print(f"  {key}: {value}")

print(f"\nTumor regions available: {list(BRATS_REGIONS.keys())}")

In [ ]:
# Extract mask pairs
if BRATS_AVAILABLE and (len(existing_pairs) == 0 or True):  # Set True to re-extract
    print("Extracting BraTS mask pairs...")
    
    extraction_summary = extract_brats_mask_pairs(
        brats_dir=BRATS_DIR,
        output_dir=MASK_PAIRS_DIR,
        region=EXTRACTION_CONFIG['region'],
        max_patients=EXTRACTION_CONFIG['max_patients'],
        slices_per_patient=EXTRACTION_CONFIG['slices_per_patient'],
        save_images=True,
        verbose=True
    )
    
    print(f"\nExtraction complete: {extraction_summary.get('total_pairs', 0)} pairs")
else:
    if not BRATS_AVAILABLE:
        print("Skipping extraction - BraTS data not available.")
    else:
        print(f"Using {len(existing_pairs)} existing pairs.")

In [ ]:
# Preview extracted pairs
all_pairs = get_all_mask_pairs(MASK_PAIRS_DIR)
print(f"Total pairs available: {len(all_pairs)}")

if all_pairs:
    print("\nSample pairs:")
    for p in all_pairs[:5]:
        print(f"  {p['patient_id']} slice {p['slice_idx']}")

In [ ]:
# Visualize sample pairs
if all_pairs:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    for i, pair in enumerate(all_pairs[:4]):
        t1_img, mask_img = load_mask_pair(
            MASK_PAIRS_DIR,
            pair['patient_id'],
            pair['slice_idx']
        )
        
        if t1_img is not None:
            axes[0, i].imshow(t1_img, cmap='gray')
            axes[0, i].set_title(f"{pair['patient_id'][:15]}\nSlice {pair['slice_idx']}")
            axes[0, i].axis('off')
            
            axes[1, i].imshow(mask_img, cmap='Reds')
            tumor_pct = np.mean(mask_img > 0) * 100
            axes[1, i].set_title(f"Tumor Mask ({tumor_pct:.1f}%)")
            axes[1, i].axis('off')
    
    plt.suptitle('Sample T1 Images and Tumor Masks', fontsize=14)
    plt.tight_layout()
    save_figure(fig, 'sample_mask_pairs.png')
    plt.show()

## 4. Generate Error Maps (Simulated)

In a full implementation, this would load the ECNN model and compute actual reconstruction error maps. For demonstration, we simulate error maps with higher values in tumor regions.

In [ ]:
# NOTE: In practice, you would:
# 1. Load the ECNN model using ecnn_model_loader
# 2. Run inference on each T1 slice
# 3. Compute pixel-wise reconstruction error

# For demonstration, we'll simulate error maps
def simulate_error_map(t1_img: np.ndarray, mask_img: np.ndarray) -> np.ndarray:
    """
    Simulate a reconstruction error map for demonstration.
    In practice, this would be computed from model predictions.
    """
    # Base noise
    error = np.random.rand(*t1_img.shape) * 0.3
    
    # Higher error in tumor regions (simulating model behavior)
    tumor_mask = mask_img > 0
    error[tumor_mask] += 0.4 + np.random.rand(tumor_mask.sum()) * 0.2
    
    # Some noise outside tumor (false positives)
    fp_mask = np.random.rand(*t1_img.shape) < 0.05
    error[fp_mask] += 0.3
    
    # Normalize
    error = np.clip(error, 0, 1)
    
    return error

print("Error map simulation function defined.")
print("Note: Replace with actual model inference for real evaluation.")

## 5. Compute Localization Metrics

In [ ]:
# Run localization evaluation on all pairs
if all_pairs:
    results = []
    
    for i, pair in enumerate(all_pairs):
        print(f"Processing {i+1}/{len(all_pairs)}: {pair['patient_id']} slice {pair['slice_idx']}", end="\r")
        
        # Load pair
        t1_img, mask_img = load_mask_pair(
            MASK_PAIRS_DIR,
            pair['patient_id'],
            pair['slice_idx']
        )
        
        if t1_img is None:
            continue
        
        # Simulate error map (replace with actual model inference)
        error_map = simulate_error_map(t1_img, mask_img)
        
        # Compute metrics
        metrics = compute_all_pixel_metrics(error_map, mask_img)
        metrics['patient_id'] = pair['patient_id']
        metrics['slice_idx'] = pair['slice_idx']
        
        results.append(metrics)
    
    print(f"\nProcessed {len(results)} samples.")
else:
    print("No pairs available for evaluation.")
    results = []

In [ ]:
# Display results as DataFrame
if results:
    df_results = pd.DataFrame(results)
    
    # Select key columns
    display_cols = ['patient_id', 'slice_idx', 'dice', 'iou', 
                   'pixel_precision', 'pixel_recall', 'pixel_f1',
                   'pixel_auroc', 'pixel_auprc']
    
    print("Per-Sample Localization Metrics:")
    display(df_results[display_cols].round(4))

In [ ]:
# Compute summary statistics
if results:
    metric_cols = ['dice', 'iou', 'pixel_precision', 'pixel_recall', 
                  'pixel_f1', 'pixel_auroc', 'pixel_auprc']
    
    summary = df_results[metric_cols].describe().round(4)
    
    print("\nSummary Statistics:")
    display(summary)
    
    # Save summary
    save_csv(summary, 'localization_summary_stats.csv')

## 6. Visualize Results

In [ ]:
# Visualize metric distributions
if results:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    metrics_to_plot = [
        ('dice', 'Dice Coefficient'),
        ('iou', 'IoU (Jaccard)'),
        ('pixel_precision', 'Pixel Precision'),
        ('pixel_recall', 'Pixel Recall'),
        ('pixel_f1', 'Pixel F1'),
        ('pixel_auroc', 'Pixel AUROC'),
        ('pixel_auprc', 'Pixel AUPRC'),
        ('true_coverage', 'True Tumor Coverage')
    ]
    
    for ax, (metric, title) in zip(axes.flat, metrics_to_plot):
        values = df_results[metric].dropna()
        ax.hist(values, bins=15, color='steelblue', alpha=0.7, edgecolor='white')
        ax.axvline(values.mean(), color='red', linestyle='--', label=f'Mean: {values.mean():.3f}')
        ax.set_xlabel(title)
        ax.set_ylabel('Count')
        ax.legend()
    
    plt.suptitle('Distribution of Localization Metrics', fontsize=14)
    plt.tight_layout()
    save_figure(fig, 'localization_metric_distributions.png')
    plt.show()

In [ ]:
# Visualize best and worst localization examples
if results:
    # Sort by Dice score
    df_sorted = df_results.sort_values('dice', ascending=False)
    
    # Get best and worst
    best_samples = df_sorted.head(3)
    worst_samples = df_sorted.tail(3)
    
    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    
    # Best samples
    for i, (_, row) in enumerate(best_samples.iterrows()):
        t1_img, mask_img = load_mask_pair(MASK_PAIRS_DIR, row['patient_id'], int(row['slice_idx']))
        if t1_img is not None:
            error_map = simulate_error_map(t1_img, mask_img)
            
            # Show T1 with mask overlay
            axes[0, i*2].imshow(t1_img, cmap='gray')
            axes[0, i*2].imshow(mask_img, cmap='Reds', alpha=0.4)
            axes[0, i*2].set_title(f"Best #{i+1}\nDice: {row['dice']:.3f}")
            axes[0, i*2].axis('off')
            
            # Show error map
            axes[0, i*2+1].imshow(error_map, cmap='hot')
            axes[0, i*2+1].set_title('Error Map')
            axes[0, i*2+1].axis('off')
    
    # Worst samples
    for i, (_, row) in enumerate(worst_samples.iterrows()):
        t1_img, mask_img = load_mask_pair(MASK_PAIRS_DIR, row['patient_id'], int(row['slice_idx']))
        if t1_img is not None:
            error_map = simulate_error_map(t1_img, mask_img)
            
            axes[1, i*2].imshow(t1_img, cmap='gray')
            axes[1, i*2].imshow(mask_img, cmap='Reds', alpha=0.4)
            axes[1, i*2].set_title(f"Worst #{i+1}\nDice: {row['dice']:.3f}")
            axes[1, i*2].axis('off')
            
            axes[1, i*2+1].imshow(error_map, cmap='hot')
            axes[1, i*2+1].set_title('Error Map')
            axes[1, i*2+1].axis('off')
    
    plt.suptitle('Best and Worst Localization Examples', fontsize=14)
    plt.tight_layout()
    save_figure(fig, 'localization_best_worst_examples.png')
    plt.show()

In [ ]:
# Detailed visualization for a single sample
if results:
    # Pick a mid-performance sample
    median_idx = len(df_sorted) // 2
    sample = df_sorted.iloc[median_idx]
    
    t1_img, mask_img = load_mask_pair(MASK_PAIRS_DIR, sample['patient_id'], int(sample['slice_idx']))
    
    if t1_img is not None:
        error_map = simulate_error_map(t1_img, mask_img)
        pred_mask = (error_map > sample['threshold_used']).astype(np.uint8)
        
        visualize_localization_result(
            original=t1_img,
            mask=mask_img,
            error_map=error_map,
            pred_mask=pred_mask,
            metrics=sample.to_dict(),
            save_path='detailed_localization_example.png'
        )

## 7. Correlation Analysis

In [ ]:
# Analyze relationship between tumor size and localization quality
if results:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Dice vs tumor coverage
    axes[0].scatter(df_results['true_coverage'], df_results['dice'], alpha=0.6)
    axes[0].set_xlabel('Tumor Coverage (fraction of image)')
    axes[0].set_ylabel('Dice Coefficient')
    axes[0].set_title('Localization vs Tumor Size')
    
    # Compute correlation
    corr = df_results['true_coverage'].corr(df_results['dice'])
    axes[0].annotate(f'r = {corr:.3f}', xy=(0.05, 0.95), xycoords='axes fraction')
    
    # IoU vs tumor coverage
    axes[1].scatter(df_results['true_coverage'], df_results['iou'], alpha=0.6, color='orange')
    axes[1].set_xlabel('Tumor Coverage')
    axes[1].set_ylabel('IoU')
    axes[1].set_title('IoU vs Tumor Size')
    
    # AUROC vs tumor coverage
    axes[2].scatter(df_results['true_coverage'], df_results['pixel_auroc'], alpha=0.6, color='green')
    axes[2].set_xlabel('Tumor Coverage')
    axes[2].set_ylabel('Pixel AUROC')
    axes[2].set_title('AUROC vs Tumor Size')
    
    plt.tight_layout()
    save_figure(fig, 'localization_tumor_size_correlation.png')
    plt.show()

## 8. Save Final Results

In [ ]:
# Generate final report
if results:
    report = {
        'timestamp': datetime.now().isoformat(),
        'num_samples': len(results),
        'tumor_region': EXTRACTION_CONFIG['region'],
        'metrics': {
            'dice': {
                'mean': float(df_results['dice'].mean()),
                'std': float(df_results['dice'].std()),
                'median': float(df_results['dice'].median())
            },
            'iou': {
                'mean': float(df_results['iou'].mean()),
                'std': float(df_results['iou'].std())
            },
            'pixel_auroc': {
                'mean': float(df_results['pixel_auroc'].mean()),
                'std': float(df_results['pixel_auroc'].std())
            },
            'pixel_auprc': {
                'mean': float(df_results['pixel_auprc'].mean()),
                'std': float(df_results['pixel_auprc'].std())
            }
        },
        'note': 'Results based on simulated error maps. Replace with actual model inference for final evaluation.'
    }
    
    # Save report
    save_json(report, 'localization_final_report.json')
    
    # Save full results
    save_csv(df_results, 'localization_full_results.csv')
    
    print("Final Localization Report:")
    print("=" * 50)
    print(f"Samples evaluated: {report['num_samples']}")
    print(f"Tumor region: {report['tumor_region']}")
    print("\nKey Metrics:")
    print(f"  Dice: {report['metrics']['dice']['mean']:.4f} +/- {report['metrics']['dice']['std']:.4f}")
    print(f"  IoU: {report['metrics']['iou']['mean']:.4f} +/- {report['metrics']['iou']['std']:.4f}")
    print(f"  AUROC: {report['metrics']['pixel_auroc']['mean']:.4f} +/- {report['metrics']['pixel_auroc']['std']:.4f}")
    print(f"  AUPRC: {report['metrics']['pixel_auprc']['mean']:.4f} +/- {report['metrics']['pixel_auprc']['std']:.4f}")
    print("=" * 50)

In [ ]:
# Generate markdown table for dissertation
if results:
    md_lines = [
        '# Localization Evaluation Results\n',
        f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}\n',
        '\n## Summary Metrics\n',
        '| Metric | Mean | Std | Median |',
        '|--------|------|-----|--------|'
    ]
    
    for metric in ['dice', 'iou', 'pixel_precision', 'pixel_recall', 'pixel_f1', 'pixel_auroc', 'pixel_auprc']:
        values = df_results[metric].dropna()
        md_lines.append(f'| {metric} | {values.mean():.4f} | {values.std():.4f} | {values.median():.4f} |')
    
    md_lines.append('\n## Notes\n')
    md_lines.append('- Results based on simulated reconstruction error maps')
    md_lines.append('- Replace with actual ECNN model inference for final evaluation')
    md_lines.append(f'- Evaluated on {len(results)} BraTS slices with tumor masks')
    
    md_content = '\n'.join(md_lines)
    
    md_path = EVALUATIONS_ROOT / 'tables' / 'localization_results.md'
    md_path.parent.mkdir(parents=True, exist_ok=True)
    with open(md_path, 'w') as f:
        f.write(md_content)
    
    print(f"Markdown table saved to: {md_path}")
    print("\n" + md_content)

---

## Conclusion

This notebook has demonstrated the localization evaluation pipeline for SymAD-ECNN:

1. **Mask Extraction**: Extracted tumor masks from BraTS segmentation volumes
2. **Error Map Generation**: (Simulated) Generated pixel-wise reconstruction error maps
3. **Metric Computation**: Computed Dice, IoU, Precision, Recall, F1, AUROC, AUPRC
4. **Visualization**: Created distribution plots and example visualizations
5. **Analysis**: Examined relationship between tumor size and localization quality

**Important**: The error maps in this notebook are simulated. For the actual dissertation evaluation, replace the `simulate_error_map` function with real ECNN model inference.

Results have been saved to the evaluations folder for use in dissertation Chapter 8.5.